<a href="https://colab.research.google.com/github/iamtrask/abcGPT/blob/main/notebooks/train_diff_logging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# abcGPT: per-pass alternating trainer with per-iter weight-diff logging

This notebook trains the small character-level shakespeare+wiki abcGPT model on a Colab GPU. Two artifacts land in `MyDrive/abcGPT/runs/<timestamp>/`:

1. **Pass snapshots** — full fp32 `state_dict`s saved at the end of each *pass* (a fixed-length block of iters drawn from a single corpus). With defaults (`iters_per_pass=250`, `max_iters=7500`), 30 snapshots are saved: shake -> wiki -> shake -> ... 15 of each. These are the primary attribution objects.
2. **Per-iter diffs** — a compressed `state_dict_k - state_dict_{k-1}` per iter, written for the full 7500 iters. These let you reconstruct any iter's state and run "how far back can attribution reach" sweeps.

Defaults: lossless fp32 diffs (zstd level 3), `first_pass_corpus='shake'`. With the 10.7M-param model the snapshots are ~43 MB each, ~1.3 GB total. Per-iter diffs dominate disk cost: each is ~35 MB lossless, ~260 GB for the run. Set `--quantize_diffs=True` to swap to int8 per-tensor diffs (~9 MB each, ~70 GB total, lossy; pass snapshots stay fp32 so rolling error can't accumulate). The trainer prints a running disk total every 100 iters so you can ctrl-C if the projection blows your quota.

Runtime: pick GPU (T4 is fine), High-RAM optional. The script captures CPU fp32 snapshots between steps so you trade a few seconds of overhead per iter for the diffs.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install dependencies

`torch` / `numpy` ship with Colab; `zstandard` does not.

In [ ]:
!pip install --quiet zstandard tiktoken

## 3. Clone abcGPT (shallow)

In [ ]:
import os
if os.path.isdir('/content/abcGPT'):
    # always pull latest main so a previously-cached clone doesn't ship stale code
    !cd /content/abcGPT && git fetch --depth 1 origin main && git reset --hard origin/main
else:
    !git clone --depth 1 --shallow-submodules https://github.com/iamtrask/abcGPT.git /content/abcGPT
%cd /content/abcGPT


## 4. Prepare the shakespeare_wiki_char dataset

Downloads tinyshakespeare and the first ~1.5 MB of wiki-abc, normalizes shakespeare to wiki surface form, concatenates with a `\n\n===\n\n` separator, and writes `train.bin` / `val.bin` / `meta.pkl`. The trainer partitions `train.bin` at that separator to serve wiki-only and shake-only batches; within a pass, every batch comes from the same corpus.

In [ ]:
!python data/shakespeare_wiki_char/prepare.py

## 5. Decide the run directory on Drive

A fresh `RUN_ID` is picked from the current timestamp on first run. If a Colab session dies mid-training, **re-mount Drive, set `RUN_ID = '<existing-dir>'` in this cell, and re-run the training cell** — the trainer detects existing pass snapshots in `out_dir` and resumes from the last completed pass (rolling optimizer state is saved alongside each snapshot for clean continuation).


In [ ]:
import time, os, glob

DRIVE_ROOT = '/content/drive/MyDrive/abcGPT/runs'
os.makedirs(DRIVE_ROOT, exist_ok=True)

# To resume an existing run, set RUN_ID = '<existing-dir>' before this cell.
# Otherwise a fresh timestamp is used.
try:
    RUN_ID
except NameError:
    RUN_ID = time.strftime('%Y%m%d-%H%M%S')

OUT_DIR = f'{DRIVE_ROOT}/{RUN_ID}'
os.makedirs(OUT_DIR, exist_ok=True)
print('run dir:', OUT_DIR)

snaps = sorted(glob.glob(os.path.join(OUT_DIR, 'pass_[0-9]*_*.pt.zst')))
if snaps:
    print(f'  found {len(snaps)} existing pass snapshots; trainer will resume')
    print(f'  latest: {os.path.basename(snaps[-1])}')
else:
    print('  fresh run (no existing pass snapshots)')

# list other runs the user might want to resume
other_runs = sorted([d for d in os.listdir(DRIVE_ROOT)
                     if d != RUN_ID and os.path.isdir(os.path.join(DRIVE_ROOT, d))])
resumable = []
for d in other_runs:
    n = len(glob.glob(os.path.join(DRIVE_ROOT, d, 'pass_[0-9]*_*.pt.zst')))
    if n > 0:
        resumable.append((d, n))
if resumable:
    print('\nother runs available (set RUN_ID = \'<name>\' to resume):')
    for d, n in resumable:
        print(f"  {d}   ({n} pass snapshots)")


## 6. Sanity check: multi-pass round-trip on CPU

Runs a tiny GPT through 3 passes x 2 iters = 6 iters, saves pass snapshots and per-iter diffs, and checks (a) each saved snapshot matches in-memory state and (b) per-iter diffs roll up correctly between snapshots. Should print `OVERALL: PASS`.

In [ ]:
!python train_diff_logging.py --verify_roundtrip=True

## 7. Train

This is the long-running cell. The trainer first does a 5-iter warmup that prints step time vs diff-capture overhead. If overhead exceeds 20%, the trainer will suggest raising `--save_every`. Then it trains for 7500 iters by default in passes of 250 iters each, alternating shake -> wiki -> shake -> ..., writes a `diff_NNNNNN.pt.zst` per iter, a `pass_PPPP_<corpus>.pt.zst` at the end of each pass, plus `iter_log.jsonl` and `pass_log.jsonl`.

Knobs:
  - `--iters_per_pass=250` (smaller = more frequent alternation, more snapshots)
  - `--first_pass_corpus=shake` (or `wiki`)
  - `--save_every=1` (set to 2 or 4 if I/O is the bottleneck)
  - `--quantize_diffs=True` (int8 diffs, ~4x smaller, lossy; pass snapshots stay fp32)

In [ ]:
!python train_diff_logging.py config/train_shakespeare_wiki_char.py \
    --out_dir=$OUT_DIR \
    --iters_per_pass=73 \
    --first_pass_corpus=shake \
    --save_diffs=False


## 8. Listing + per-pass table

Lists the pass snapshots and per-iter diffs that landed on Drive, parses `pass_log.jsonl`, and prints `pass_idx | corpus | val_loss | snapshot_size`.

In [ ]:
import os, glob, json
diffs = sorted(glob.glob(os.path.join(OUT_DIR, 'diff_*.pt.zst')))
snaps = sorted(glob.glob(os.path.join(OUT_DIR, 'pass_*.pt.zst')))
iter_log = os.path.join(OUT_DIR, 'iter_log.jsonl')
pass_log = os.path.join(OUT_DIR, 'pass_log.jsonl')
meta = os.path.join(OUT_DIR, 'run_meta.json')

def total(paths):
    return sum(os.path.getsize(p) for p in paths)

diff_bytes = total(diffs)
snap_bytes = total(snaps)
print(f'run dir:         {OUT_DIR}')
print(f'pass snapshots:  {len(snaps)} files, {snap_bytes/1e9:.2f} GB')
print(f'per-iter diffs:  {len(diffs)} files, {diff_bytes/1e9:.2f} GB')
print(f'iter_log.jsonl:  exists={os.path.exists(iter_log)}')
print(f'pass_log.jsonl:  exists={os.path.exists(pass_log)}')
print(f'run_meta.json:   exists={os.path.exists(meta)}')
print(f'TOTAL:           {(diff_bytes+snap_bytes)/1e9:.2f} GB')

# per-pass table
if os.path.exists(pass_log):
    print()
    print(f"{'pass':>4} {'corpus':>6} {'val_loss':>10} {'snap_MB':>10}")
    print('-' * 36)
    with open(pass_log) as f:
        for line in f:
            row = json.loads(line)
            print(f"{row['pass_idx']:>4d} {row['corpus']:>6s} "
                  f"{row['val_loss_at_pass_end']:>10.4f} "
                  f"{row['snapshot_size_bytes']/1e6:>10.2f}")

## 9. Optional: reconstruct any iter's `state_dict`

Pick a target iter; load the nearest preceding pass snapshot (or `pass_init.pt.zst` if you want pre-pass-0 state), then apply per-iter diffs forward.

In [ ]:
import io, glob, os, re, zstandard, torch, json

def _decompress(b):
    return zstandard.ZstdDecompressor().decompress(b)

def _load_pt_zst(path):
    with open(path, 'rb') as f:
        blob = _decompress(f.read())
    return torch.load(io.BytesIO(blob), map_location='cpu', weights_only=False)

def load_state_at_iter(out_dir, target_iter):
    """Reconstruct state_dict at target_iter by starting from the nearest
    preceding pass snapshot (or pass_init for iters before pass 0 ends)
    and applying per-iter diffs forward."""
    # pass_PPPP_<corpus>.pt.zst — parse pass index from filename
    snap_paths = sorted(glob.glob(os.path.join(out_dir, 'pass_*.pt.zst')))
    candidates = []
    with open(os.path.join(out_dir, 'run_meta.json')) as f:
        meta = json.load(f)
    iters_per_pass = meta['iters_per_pass']
    for p in snap_paths:
        m = re.search(r'pass_(\d+)_', os.path.basename(p))
        if not m:
            continue
        pass_idx = int(m.group(1))
        end_iter = (pass_idx + 1) * iters_per_pass
        if end_iter <= target_iter:
            candidates.append((end_iter, p))
    if candidates:
        start_iter, snap_path = max(candidates)
    else:
        snap_path = os.path.join(out_dir, 'pass_init.pt.zst')
        start_iter = 0
    sd = _load_pt_zst(snap_path)['state_dict']
    for i in range(start_iter + 1, target_iter + 1):
        diff_path = os.path.join(out_dir, f'diff_{i:06d}.pt.zst')
        if not os.path.exists(diff_path):
            continue
        payload = _load_pt_zst(diff_path)
        diff = payload['diff']
        if payload.get('quantized'):
            diff = {k: v['q'].to(torch.float32) * v['scale'] for k, v in diff.items()}
        sd = {k: sd[k] + diff[k] for k in sd}
    return sd

# example: reconstruct iter 750
# sd = load_state_at_iter(OUT_DIR, 750)
# print({k: v.shape for k, v in list(sd.items())[:3]})

## 10. Weighted attribution sampling

Pick a base pass `K`. Above K, the model is treated as a shared base. After K, the per-pass diffs are split by which corpus that pass trained on. Scale the shake diffs by `α` and the wiki diffs by `β`, sum them onto `S_K`, and sample from the resulting model.

- `α = 1, β = 0` &rarr; "100% shakespeare" (only shake diffs after K are applied)
- `α = 0, β = 1` &rarr; "100% wikipedia"
- `α = β = 1` &rarr; reconstructs `S_N` exactly
- `α = β = 0` &rarr; just samples from `S_K`
- Negative coefficients are unlearning experiments (subtract a corpus's cumulative contribution)

When `K` is too small the early diffs are unstable and the weighted composition goes off-manifold (output noisy or collapsed). When `K` is too large the diffs are tiny and the slider barely moves the model. The interesting regime is in the middle.

In [ ]:
import io, glob, os, json, pickle, zstandard, torch
from ipywidgets import FloatSlider, IntSlider, Text, Button, Output, VBox
from IPython.display import display
from model import GPT, GPTConfig

# --- load per-pass snapshots + run metadata ---
RUN_META = json.load(open(os.path.join(OUT_DIR, 'run_meta.json')))
PASS_LOG = [json.loads(l) for l in open(os.path.join(OUT_DIR, 'pass_log.jsonl'))]

def _zload(path):
    with open(path, 'rb') as f:
        blob = zstandard.ZstdDecompressor().decompress(f.read())
    return torch.load(io.BytesIO(blob), map_location='cpu', weights_only=False)

snap_paths = [os.path.join(OUT_DIR, 'pass_init.pt.zst')] + sorted(glob.glob(os.path.join(OUT_DIR, 'pass_*.pt.zst')))
snap_paths = list(dict.fromkeys(snap_paths))   # de-dupe in case pass_init matched the glob
snaps   = [_zload(p)['state_dict'] for p in snap_paths]
# corpora[i] = corpus that produced the diff snaps[i-1] -> snaps[i]
corpora = [None] + [row['corpus'] for row in PASS_LOG]
print(f'{len(snaps)} snapshots loaded ({len(PASS_LOG)} passes; corpora: {sorted(set(c for c in corpora if c))})')

# --- precompute per-pass diffs ---
diffs = [None] + [{k: snaps[i][k] - snaps[i-1][k] for k in snaps[i]} for i in range(1, len(snaps))]

def weighted_state(K, alpha_shake, beta_wiki):
    """state = S_K + alpha * sum(shake-pass diffs after K) + beta * sum(wiki-pass diffs after K)"""
    state = {k: snaps[K][k].clone() for k in snaps[K]}
    for i in range(K + 1, len(snaps)):
        coef = alpha_shake if corpora[i] == 'shake' else beta_wiki
        if coef == 0.0:
            continue
        for k in diffs[i]:
            state[k] = state[k] + coef * diffs[i][k]
    return state

# --- model rebuild + char vocab from prepare.py output ---
META_PKL = pickle.load(open('data/shakespeare_wiki_char/meta.pkl', 'rb'))
stoi, itos = META_PKL['stoi'], META_PKL['itos']

gptconf = GPTConfig(**RUN_META['model_args'])
device  = 'cuda' if torch.cuda.is_available() else 'cpu'
model   = GPT(gptconf).eval().to(device)

def generate_from_state(state, prompt, max_new_tokens=300, temperature=0.8, top_k=40, seed=42):
    model.load_state_dict({k: v.to(device) for k, v in state.items()})
    torch.manual_seed(seed)
    ids = [stoi[c] for c in prompt if c in stoi] or [0]
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    with torch.no_grad():
        out = model.generate(idx, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    return ''.join(itos.get(i.item(), '?') for i in out[0])

# --- interactive widget ---
N = len(snaps) - 1
K_slider     = IntSlider(value=N // 4, min=0, max=N, step=1, description='base K')
alpha_slider = FloatSlider(value=1.0, min=-1.0, max=2.0, step=0.05, description='alpha (shake)')
beta_slider  = FloatSlider(value=0.0, min=-1.0, max=2.0, step=0.05, description='beta (wiki)')
prompt_box   = Text(value='HAMLET:\n', description='prompt')
btn          = Button(description='Generate', button_style='primary')
out_widget   = Output(layout={'border': '1px solid #ddd', 'padding': '6px', 'width': '780px'})

def on_click(_=None):
    with out_widget:
        out_widget.clear_output()
        state = weighted_state(K_slider.value, alpha_slider.value, beta_slider.value)
        text  = generate_from_state(state, prompt_box.value)
        print(text)
        print(f'\n[K={K_slider.value} | alpha={alpha_slider.value:.2f} | beta={beta_slider.value:.2f} | seed=42]')

btn.on_click(on_click)
display(VBox([K_slider, alpha_slider, beta_slider, prompt_box, btn, out_widget]))
on_click()


## 11. Diff orthogonality across passes

If shake-passes and wiki-passes update approximately orthogonal subspaces of the model's weights, the (α, β) knobs above behave like independent dials and the per-source attribution is clean. If they share substantial overlap, turning α down quietly breaks β-controlled behavior. The cosine plot below makes this empirically visible across training.

- **Near 0**: diffs are disjoint, attribution is clean.
- **Near 1**: diffs are stepping on each other, attribution is mush.

Plotted is the cosine between each shake-pass diff and the wiki-pass diff that immediately followed (or preceded) it.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def _flat(d):
    return torch.cat([v.flatten() for v in d.values()])

def _cos(a, b):
    fa, fb = _flat(a), _flat(b)
    return (fa @ fb).item() / (fa.norm().item() * fb.norm().item() + 1e-12)

xs, ys = [], []
for i in range(1, len(diffs) - 1):
    if corpora[i] != corpora[i + 1]:
        xs.append(i)
        ys.append(_cos(diffs[i], diffs[i + 1]))

plt.figure(figsize=(11, 4))
plt.plot(xs, ys, marker='o', linewidth=1.2)
plt.axhline(0, color='gray', linewidth=0.6)
plt.xlabel('pass index')
plt.ylabel('cos(shake-diff, wiki-diff)')
plt.title('Adjacent shake / wiki diff cosine across training')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'mean cosine: {np.mean(ys):.4f}   median: {np.median(ys):.4f}   range: [{min(ys):.4f}, {max(ys):.4f}]')
